In [ ]:
import os
import sys
import numpy as np
from ase.io import read, write

from mace.cli.run_train import main

# Define LOCAL file paths inside Colab
outcar_file = "/content/OUTCAR"

data_dir = "/content/inter3_data"
my_sim_checkpoints_dir = "/content/inter3_checkpoints"
my_sim_results_dir = "/content/inter3_results"

os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

# Start at frame 6840 and take every 5th frame
print("Reading and subsampling OUTCAR...")
frames = read(outcar_file, index="6840::10", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")

# FIX ASE KEYS
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

my_atomic_numbers = "[1, 6, 7, 50, 53]"

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts_inter3",
    f"--train_file={train_file}",
    f"--valid_file={valid_file}",

    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=average",

    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=200",
    "--patience=10",
    "--batch_size=5",
    "--valid_batch_size=5",
    "--device=cuda",
    "--default_dtype=float32",

    "--r_max=5.0",
    "--correlation=3",

    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",


    "--num_interactions=3",


    f"--atomic_numbers={my_atomic_numbers}",
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0",
    "--forces_weight=100.0",

    "--optimizer=adamw",
    "--amsgrad",
    "--weight_decay=5e-7",

    "--ema",
    "--ema_decay=0.99",

    "--swa",
    "--start_swa=30",

    "--swa_forces_weight=100.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",

    "--scaling=rms_forces_scaling",
    "--restart_latest"
]

sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

Reading and subsampling OUTCAR...
Extracted 316 distinct frames.
Renaming ASE internal keys...


INFO:root:===========VERIFYING SETTINGS===========


Saved 252 train, 31 valid, and 33 test frames.
2026-09-21 12:50:08.379 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 12:50:08.379 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 12:50:08.379 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


2026-09-21 12:50:08.382 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 12:50:08.382 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 12:50:08.382 INFO: Stage Two is activated as start_stage_two was defined


...

2026-09-21 13:03:16.582 INFO: Evaluating valid_Default ...
2026-09-21 13:03:16.582 INFO: Evaluating valid_Default ...
2026-09-21 13:03:16.582 INFO: Evaluating valid_Default ...


INFO:root:Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.5      |         45.0     |         13.37     |
| valid_Default |            1.6      |         54.2     |         15.65     |
+---------------+---------------------+------------------+-------------------+


2026-09-21 13:03:16.957 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.5      |         45.0     |         13.37     |
| valid_Default |            1.6      |         54.2     |         15.65     |
+---------------+---------------------+------------------+-------------------+
2026-09-21 13:03:16.957 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.5      |         45.0     |         13.37     |
| valid_Default |            1.6      |         54.2     |         15.65     |
+------

DEBUG:root:Running inference on train_Default dataset
DEBUG:root:Running inference on valid_Default dataset
INFO:root:Done


2026-09-21 13:03:25.953 INFO: Done
2026-09-21 13:03:25.953 INFO: Done
2026-09-21 13:03:25.953 INFO: Done
Training completed!


Simulating trajectory with ase

In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

drive.mount('/content/drive')

model_path = "model_3000_pts_inter3_stagetwo_compiled.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/numinter/md_trajectory_inter3_2000.extxyz"

output_dir = os.path.dirname(output_traj_path)
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured output directory exists: {output_dir}")


#  Load the initial structure
print("Loading initial structure...")
atoms = read(init_structure_path, index=0)

# Initialize the MACE Calculator with your compiled model
print(f"Loading compiled MACE model from: {model_path}...")
calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator

# Set up the MD Simulation (NVT Ensemble at 300K)
temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

# Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=1)

# Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured output directory exists: /content/drive/MyDrive/MDProject/results/numinter
Loading initial structure...
Loading compiled MACE model from: model_3000_pts_inter3_stagetwo_compiled.model...


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  torch.load(f=model_path, map_location=device)


Initializing velocities at 300 K...

Starting MD simulation for 2000 steps...


/tmp/ipykernel_2606/3766940398.py:43: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    0 | Temp:  295.2 K | Epot: -479.159 eV | Total E: -475.343 eV
Step:    1 | Temp:  272.0 K | Epot: -478.822 eV | Total E: -475.307 eV
Step:    2 | Temp:  225.6 K | Epot: -478.178 eV | Total E: -475.262 eV
Step:    3 | Temp:  199.2 K | Epot: -477.823 eV | Total E: -475.249 eV
Step:    4 | Temp:  206.1 K | Epot: -477.937 eV | Total E: -475.274 eV
Step:    5 | Temp:  224.5 K | Epot: -478.207 eV | Total E: -475.305 eV
Step:    6 | Temp:  229.7 K | Epot: -478.270 eV | Total E: -475.301 eV
Step:    7 | Temp:  216.1 K | Epot: -478.084 eV | Total E: -475.291 eV
Step:    8 | Temp:  205.6 K | Epot: -477.937 eV | Total E: -475.279 eV
Step:    9 | Temp:  223.1 K | Epot: -478.165 eV | Total E: -475.280 eV
Step:   10 | Temp:  267.2 K | Epot: -478.752 eV | Total E: -475.298 eV
Step:   11 | Temp:  303.5 K | Epot: -479.261 eV | Total E: -475.339 eV
Step:   12 | Temp:  306.2 K | Epot: -479.293 eV | Total E: -475.335 eV
Step:   13 | Temp:  283.4 K | Epot: -478.962 eV | Total E: -475.299 eV
Step: 

Full analysis of the trajectory

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/numinter/md_trajectory_inter3_2000.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/numInt/int3"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



[Total Energy] t0 = 1065 (frame 1065), g = 15.51, N_eff = 60.4 / 2001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.8066
Area (180° peak): 0.1934
Max v dataset: 0.14017246
Processed 2001 frames successfully.
Processed 2001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 1250.67 K


In [ ]:
! pip install ase
! pip install pymbar
! pip install mace-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.1/453.1 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.8 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39523 sha256=d2aa79f5b037df19665ecbe37d6b0a3b37e168b1a5cefad8f2068431ffa21feb
  Stored in directory: /root/.cache/pip/wheels/05/2e/7e/39f980a5a0755f

In [ ]:
import shutil
from google.colab import files

print("Zipping all checkpoints and models...")
# Compress the entire checkpoints folder into one zip file
shutil.make_archive('inter3_backuptrue', 'zip', '/content/inter3_checkpoints')

print("Downloading to your computer...")
# Automatically trigger the browser download
files.download('inter3_backuptrue.zip')

Zipping all checkpoints and models...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

print("Zipping all checkpoints and models...")
# Compress the entire checkpoints folder into one zip file
shutil.make_archive('inter3_datatrue', 'zip', '/content/inter3_data')

print("Downloading to your computer...")
# Automatically trigger the browser download
files.download('inter3_datatrue.zip')

Zipping all checkpoints and models...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

print("Zipping all checkpoints and models...")
# Compress the entire checkpoints folder into one zip file
shutil.make_archive('inter3_resultstrue', 'zip', '/content/inter3_results')

print("Downloading to your computer...")
# Automatically trigger the browser download
files.download('inter3_resultstrue.zip')

Zipping all checkpoints and models...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

print("Zipping all checkpoints and models...")
# Compress the entire checkpoints folder into one zip file
shutil.make_archive('inter3_logstrue', 'zip', '/content/logs')

print("Downloading to your computer...")
# Automatically trigger the browser download
files.download('inter3_logstrue.zip')

Zipping all checkpoints and models...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## --num_interactrion=4

In [ ]:
import os
import sys
import numpy as np
from ase.io import read, write

from mace.cli.run_train import main

outcar_file = "/content/OUTCAR"

data_dir = "/content/inter4_data2"
my_sim_checkpoints_dir = "/content/inter4_checkpoints2"
my_sim_results_dir = "/content/inter4_results2"

os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

# Start at frame 6840 and take every 10th frame
print("Reading and subsampling OUTCAR...")
frames = read(outcar_file, index="6840::10", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")

# FIX ASE KEYS
print("Renaming ASE internal keys...")
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

my_atomic_numbers = "[1, 6, 7, 50, 53]"

custom_args = [
    "mace_run_training",
    "--name=MODEL_3000_pts_inter4_2",
    f"--train_file={train_file}",
    f"--valid_file={valid_file}",

    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=average",

    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=200",
    "--patience=10",
    "--batch_size=5",
    "--valid_batch_size=5",
    "--device=cuda",
    "--default_dtype=float32",

    "--r_max=5.0",
    "--correlation=3",

    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",


    "--num_interactions=4",


    f"--atomic_numbers={my_atomic_numbers}",
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0",
    "--forces_weight=100.0",

    "--optimizer=adamw",
    "--amsgrad",
    "--weight_decay=5e-7",

    "--ema",
    "--ema_decay=0.99",

    "--swa",
    "--start_swa=30",

    "--swa_forces_weight=100.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",

    "--scaling=rms_forces_scaling",
    "--restart_latest"
]

sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

Reading and subsampling OUTCAR...
Extracted 316 distinct frames.
Renaming ASE internal keys...


INFO:root:===========VERIFYING SETTINGS===========


Saved 252 train, 31 valid, and 33 test frames.
2026-09-21 13:52:25.199 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 13:52:25.199 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 13:52:25.199 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 13:52:25.199 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 13:52:25.199 INFO: ===========VERIFYING SETTINGS===========
2026-09-21 13:52:25.199 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


2026-09-21 13:52:25.207 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 13:52:25.207 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 13:52:25.207 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 13:52:25.207 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 13:52:25.207 INFO: Stage Two is activated as start_stage_two was defined
2026-09-21 13:52:25.207 INFO: Stage Two is activated as start_stage_two was defined


...

2026-09-21 14:25:13.843 INFO: Evaluating valid_Default ...
2026-09-21 14:25:13.843 INFO: Evaluating valid_Default ...
2026-09-21 14:25:13.843 INFO: Evaluating valid_Default ...
2026-09-21 14:25:13.843 INFO: Evaluating valid_Default ...
2026-09-21 14:25:13.843 INFO: Evaluating valid_Default ...
2026-09-21 14:25:13.843 INFO: Evaluating valid_Default ...


INFO:root:Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.3      |         43.1     |         12.82     |
| valid_Default |            1.0      |         55.4     |         15.98     |
+---------------+---------------------+------------------+-------------------+


2026-09-21 14:25:14.331 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.3      |         43.1     |         12.82     |
| valid_Default |            1.0      |         55.4     |         15.98     |
+---------------+---------------------+------------------+-------------------+
2026-09-21 14:25:14.331 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.3      |         43.1     |         12.82     |
| valid_Default |            1.0      |         55.4     |         15.98     |
+------

DEBUG:root:Running inference on train_Default dataset
DEBUG:root:Running inference on valid_Default dataset
INFO:root:Done


2026-09-21 14:25:21.945 INFO: Done
2026-09-21 14:25:21.945 INFO: Done
2026-09-21 14:25:21.945 INFO: Done
2026-09-21 14:25:21.945 INFO: Done
2026-09-21 14:25:21.945 INFO: Done
2026-09-21 14:25:21.945 INFO: Done
Training completed!


Code just to save the data locally since with an account i couldn't access drive

In [ ]:
import shutil
from google.colab import files
print("Zipping all checkpoints and models...")
# Compress the entire checkpoints folder into one zip file
shutil.make_archive('inter4_backuptrue2', 'zip', '/content/inter4_checkpoints2')

print("Downloading to your computer...")
# Automatically trigger the browser download
files.download('inter4_backuptrue2.zip')
print("Zipping all checkpoints and models...")
# Compress the entire checkpoints folder into one zip file
shutil.make_archive('inter4_datatrue2', 'zip', '/content/inter4_data2')

print("Downloading to your computer...")
# Automatically trigger the browser download
files.download('inter4_datatrue2.zip')
print("Zipping all checkpoints and models...")
# Compress the entire checkpoints folder into one zip file
shutil.make_archive('inter4_resultstrue2', 'zip', '/content/inter4_results2')

print("Downloading to your computer...")
# Automatically trigger the browser download
files.download('inter4_resultstrue2.zip')
print("Zipping all checkpoints and models...")
# Compress the entire checkpoints folder into one zip file
shutil.make_archive('inter4_logstrue2', 'zip', '/content/logs')

print("Downloading to your computer...")
# Automatically trigger the browser download
files.download('inter4_logstrue2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Zipping all checkpoints and models...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Zipping all checkpoints and models...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Simulating the trajectories with ase

In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

drive.mount('/content/drive')

model_path = "MODEL_3000_pts_inter4_2_run-1_stagetwo.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/numinter/md_trajectory_inter4_2000.extxyz"

output_dir = os.path.dirname(output_traj_path)
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured output directory exists: {output_dir}")
# -------------------------------------------------------------------------------

# 3. Load the initial structure (frame 0)
print("Loading initial structure...")
atoms = read(init_structure_path, index=0)

# Initialize the MACE Calculator with your compiled model
print(f"Loading compiled MACE model from: {model_path}...")
calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator

# Set up the MD Simulation (NVT Ensemble at 300K)
temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

# Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=1)

# Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured output directory exists: /content/drive/MyDrive/MDProject/results/numinter
Loading initial structure...
Loading compiled MACE model from: MODEL_3000_pts_inter4_2_run-1_stagetwo.model...
Initializing velocities at 300 K...

Starting MD simulation for 2000 steps...


/tmp/ipykernel_2606/2275852328.py:43: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    0 | Temp:  258.9 K | Epot: -479.038 eV | Total E: -475.692 eV
Step:    1 | Temp:  239.8 K | Epot: -478.772 eV | Total E: -475.673 eV
Step:    2 | Temp:  200.6 K | Epot: -478.239 eV | Total E: -475.645 eV
Step:    3 | Temp:  175.7 K | Epot: -477.910 eV | Total E: -475.639 eV
Step:    4 | Temp:  178.6 K | Epot: -477.967 eV | Total E: -475.658 eV
Step:    5 | Temp:  193.3 K | Epot: -478.178 eV | Total E: -475.680 eV
Step:    6 | Temp:  195.5 K | Epot: -478.211 eV | Total E: -475.684 eV
Step:    7 | Temp:  180.6 K | Epot: -477.999 eV | Total E: -475.665 eV
Step:    8 | Temp:  168.5 K | Epot: -477.820 eV | Total E: -475.642 eV
Step:    9 | Temp:  182.2 K | Epot: -478.001 eV | Total E: -475.645 eV
Step:   10 | Temp:  221.1 K | Epot: -478.521 eV | Total E: -475.664 eV
Step:   11 | Temp:  255.6 K | Epot: -478.984 eV | Total E: -475.681 eV
Step:   12 | Temp:  259.6 K | Epot: -479.031 eV | Total E: -475.675 eV
Step:   13 | Temp:  241.2 K | Epot: -478.754 eV | Total E: -475.637 eV
Step: 

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/numinter/md_trajectory_inter4_2000.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/numInt/int4"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

[Total Energy] t0 = 1057 (frame 1057), g = 43.36, N_eff = 21.8 / 2001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.8056
Area (180° peak): 0.1944
Max v dataset: 0.12716117999999987
Processed 2001 frames successfully.
Processed 2001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 1222.51 K


In [ ]:
import os
import sys
import numpy as np
from ase.io import read, write
from google.colab import drive

from mace.cli.run_train import main

# Mount Google Drive to access the MDProject folder
drive.mount('/content/drive')

# Define file paths
outcar_file = "/content/drive/MyDrive/MDProject/OUTCAR"

data_dir = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/inter2new"
my_sim_checkpoints_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/inter2new"
my_sim_results_dir = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/results/inter2new"

os.makedirs(data_dir, exist_ok=True)

train_file = f"{data_dir}/train.extxyz"
valid_file = f"{data_dir}/valid.extxyz"
test_file = f"{data_dir}/test.extxyz"

# Start at frame 6840 and take every 5th frame
print("Reading and subsampling OUTCAR...")
frames = read(outcar_file, index="6840::10", format="vasp-out")
print(f"Extracted {len(frames)} distinct frames.")

# FIX ASE KEYS
print("Renaming ASE internal keys...")
for frame in frames:
    if 'energy' in frame.info:
        frame.info['REF_energy'] = frame.info.pop('energy')
    if 'stress' in frame.info:
        frame.info['REF_stress'] = frame.info.pop('stress')
    if 'forces' in frame.arrays:
        frame.arrays['REF_forces'] = frame.arrays.pop('forces')

# SPLIT: 80% Train, 10% Valid, 10% Test
n_total = len(frames)
n_train = int(0.8 * n_total)
n_valid = int(0.1 * n_total)

train_frames = frames[:n_train]
valid_frames = frames[n_train : n_train + n_valid]
test_frames = frames[n_train + n_valid:]

# SAVE
write(train_file, train_frames, format="extxyz")
write(valid_file, valid_frames, format="extxyz")
write(test_file, test_frames, format="extxyz")

print(f"Saved {len(train_frames)} train, {len(valid_frames)} valid, and {len(test_frames)} test frames.")

my_atomic_numbers = "[1, 6, 7, 50, 53]"

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts_inter2new",
    f"--train_file={train_file}",
    f"--valid_file={valid_file}",

    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=average",

    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=200",
    "--patience=10",
    "--batch_size=5",
    "--valid_batch_size=5",
    "--device=cuda",
    "--default_dtype=float32",

    "--r_max=5.0",
    "--correlation=3",

    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",


    "--num_interactions=2",


    f"--atomic_numbers={my_atomic_numbers}",
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0",
    "--forces_weight=100.0",

    "--optimizer=adamw",
    "--amsgrad",
    "--weight_decay=5e-7",

    "--ema",
    "--ema_decay=0.99",

    "--swa",
    "--start_swa=30",

    "--swa_forces_weight=100.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",

    "--scaling=rms_forces_scaling",
    "--restart_latest"
]

sys.argv = custom_args

if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

Mounted at /content/drive
Reading and subsampling OUTCAR...
Extracted 316 distinct frames.
Renaming ASE internal keys...


INFO:root:===========VERIFYING SETTINGS===========


Saved 252 train, 31 valid, and 33 test frames.
2026-09-26 11:41:52.452 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


2026-09-26 11:41:52.456 INFO: Stage Two is activated as start_stage_two was defined


...

2026-09-26 12:01:08.582 INFO: Evaluating valid_Default ...


INFO:root:Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.9      |         46.2     |         13.75     |
| valid_Default |            1.6      |         56.1     |         16.20     |
+---------------+---------------------+------------------+-------------------+


2026-09-26 12:01:08.855 INFO: Error-table on TRAIN and VALID:
+---------------+---------------------+------------------+-------------------+
|  config_type  | RMSE E / meV / atom | RMSE F / meV / A | relative F RMSE % |
+---------------+---------------------+------------------+-------------------+
| train_Default |            1.9      |         46.2     |         13.75     |
| valid_Default |            1.6      |         56.1     |         16.20     |
+---------------+---------------------+------------------+-------------------+


DEBUG:root:Running inference on train_Default dataset
DEBUG:root:Running inference on valid_Default dataset
INFO:root:Done


2026-09-26 12:01:14.926 INFO: Done
Training completed!


In [ ]:
import os
import time
from ase.io import read, write
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
from google.colab import drive

#  Mount Google Drive
drive.mount('/content/drive')

# Define File Paths
model_path = "/content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/inter2new/model_3000_pts_inter2new_run-1_stagetwo.model"
init_structure_path = "/content/drive/MyDrive/MDProject/new_data/1.NVT_300/A.1_10K/Smart_FPS/valid_fps.extxyz"
output_traj_path = "/content/drive/MyDrive/MDProject/results/numinter/md_trajectory_inter2new_2000.extxyz"

#  Automatically create the output directory path if it doesn't exist
output_dir = os.path.dirname(output_traj_path)
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured output directory exists: {output_dir}")


# Load the initial structure (frame 0)
print("Loading initial structure...")
atoms = read(init_structure_path, index=0)

# Initialize the MACE Calculator with your compiled model
print(f"Loading compiled MACE model from: {model_path}...")
calculator = MACECalculator(
    model_paths=[model_path],
    device='cuda',
    default_dtype='float32'
)
atoms.calc = calculator

# Set up the MD Simulation (NVT Ensemble at 300K)
temperature_K = 300
timestep_fs = 1.0
friction_coeff = 0.001

print(f"Initializing velocities at {temperature_K} K...")
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)

dyn = Langevin(
    atoms,
    timestep_fs * units.fs,
    temperature_K=temperature_K,
    friction=friction_coeff
)

#  Define an output function to save frames
def write_frame():
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(atoms))

    # Append to the .extxyz trajectory file
    write(output_traj_path, atoms, append=True)

    # Print status every 10 steps
    step = dyn.get_number_of_steps()
    print(f"Step: {step:>4} | Temp: {temp:>6.1f} K | Epot: {epot:+.3f} eV | Total E: {epot+ekin:+.3f} eV")

# Clear old trajectory file if it exists
if os.path.exists(output_traj_path):
    os.remove(output_traj_path)

# Attach the logger to run every 10 steps
dyn.attach(write_frame, interval=1)

# Run the Simulation
num_steps = 2000
print(f"\nStarting MD simulation for {num_steps} steps...")
start_time = time.time()

dyn.run(num_steps)

print(f"\nMD simulation finished in {time.time() - start_time:.2f} seconds!")
print(f"Trajectory safely saved to: {output_traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured output directory exists: /content/drive/MyDrive/MDProject/results/numinter
Loading initial structure...


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
INFO:root:CUDA version: 12.8, CUDA device: 0


Loading compiled MACE model from: /content/drive/MyDrive/MDProject/new_simulation/1.NVT_300/A.1_10K/checkpoints/inter2new/model_3000_pts_inter2new_run-1_stagetwo.model...
2026-09-26 12:28:54.633 INFO: CUDA version: 12.8, CUDA device: 0


INFO:root:Using head Default out of  ['Default']


2026-09-26 12:28:54.635 INFO: Using head Default out of  ['Default']
Initializing velocities at 300 K...


/tmp/ipykernel_1070/989192320.py:43: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)



Starting MD simulation for 2000 steps...
Step:    0 | Temp:  268.9 K | Epot: -478.903 eV | Total E: -475.427 eV
Step:    1 | Temp:  250.2 K | Epot: -478.646 eV | Total E: -475.412 eV
Step:    2 | Temp:  212.3 K | Epot: -478.119 eV | Total E: -475.375 eV
Step:    3 | Temp:  188.9 K | Epot: -477.794 eV | Total E: -475.352 eV
Step:    4 | Temp:  191.2 K | Epot: -477.836 eV | Total E: -475.365 eV
Step:    5 | Temp:  202.1 K | Epot: -477.993 eV | Total E: -475.381 eV
Step:    6 | Temp:  199.0 K | Epot: -477.947 eV | Total E: -475.375 eV
Step:    7 | Temp:  182.4 K | Epot: -477.705 eV | Total E: -475.347 eV
Step:    8 | Temp:  174.2 K | Epot: -477.584 eV | Total E: -475.332 eV
Step:    9 | Temp:  195.2 K | Epot: -477.856 eV | Total E: -475.333 eV
Step:   10 | Temp:  237.0 K | Epot: -478.432 eV | Total E: -475.368 eV
Step:   11 | Temp:  272.1 K | Epot: -478.906 eV | Total E: -475.389 eV
Step:   12 | Temp:  277.7 K | Epot: -478.957 eV | Total E: -475.367 eV
Step:   13 | Temp:  259.5 K | Epot:

In [ ]:
import os
import automatic_analysis5

input_traj_dir = "/content/drive/MyDrive/MDProject/results/numinter/md_trajectory_inter2new_2000.extxyz"
output_dir = "/content/drive/MyDrive/MDProject/results/inter2new"

# Create the directory before running the analysis
os.makedirs(output_dir, exist_ok=True)

automatic_analysis5.full_analysis(input_traj_dir, output_dir, rmax=3.9, nbins=100, timestep_in_fs=0.5, window_size=2)

2026-09-26 12:30:18.411 WARNING: Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


DEBUG:jax._src.path:etils.epath found. Using etils.epath for file I/O.
****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



2026-09-26 12:30:19.068 WARNING: 
****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



INFO:pymbar.mbar_solvers:JAX detected. Using JAX acceleration.


2026-09-26 12:30:19.113 INFO: JAX detected. Using JAX acceleration.
[Total Energy] t0 = 2 (frame 2), g = 52.27, N_eff = 38.3 / 2001
N atoms (equilibrium):  100 in volume: 1474.4473736057512
Done computing for pair ('Sn', 'I')
Done computing for pair ('I', 'I')
Done computing for pair ('N', 'I')
Done computing for pair ('H', 'I')
Done computing for pair ('C', 'N')
Done computing for pair ('H', 'N')
Done computing for pair ('H', 'C')
Area (90° peak): 0.8086
Area (180° peak): 0.1914
Max v dataset: 0.14139467999999988
Processed 2001 frames successfully.
Processed 2001 frames successfully. Elements: [np.str_('C'), np.str_('H'), np.str_('I'), np.str_('N'), np.str_('Sn')]
Calculated mean temperature: 1253.21 K


In [ ]:
! pip install pymbar
! pip install ase
! pip install mace-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 44.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.5/457.5 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 8.1 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39523 sha256=70248f75391b9cfa48298a70f830303fd8473665a7a599b382e2dd19679f822d
  Stored in directory: /root/.cache/pip/wheels/05/2e/7e/39f980a5a0755f